# PlayStation Marketing MMM Hands-On Lab

**Duration**: 2 hours active (after 30min setup)  
**Approach**: Copy prompts into Cortex Code, iterate, validate with the SQL cells below  
**Your workspace**: `PS_DEMO.<YOUR_NAME>_HOL` (you create this)

---

## Setup: Load the Data (first 15 minutes)

You have a folder of CSV files containing PlayStation marketing data. Follow these steps to load them into Snowflake using Cortex Code.

### Step 1: Create your schema

In [ ]:
-- Replace YOUR_NAME with your initials (e.g., JCHEN)
CREATE SCHEMA IF NOT EXISTS PS_DEMO.YOUR_NAME_HOL;
USE SCHEMA PS_DEMO.YOUR_NAME_HOL;

### Step 2: Upload CSVs via Cortex Code

1. In Cortex Code, click the **+** (attachment/file) button in the chat input
2. Select **all CSV files** from the `data/` folder you received
3. Use this prompt to load them:

```
I've attached CSV files containing PlayStation marketing data.
Please create tables in my current schema (PS_DEMO.YOUR_NAME_HOL)
from each CSV file. Use the filename (without .csv) as the table name.
Infer column types from the data - dates should be DATE type,
numbers should be NUMBER, and text should be VARCHAR.
Load all the data from each file.
```

**Alternative** (if file attachment isn't available): Upload CSVs to a stage first:

```sql
CREATE OR REPLACE STAGE PS_DEMO.YOUR_NAME_HOL.HOL_STAGE;
-- Then use Snowsight's Upload button on the stage, or:
PUT file:///path/to/data/*.csv @PS_DEMO.YOUR_NAME_HOL.HOL_STAGE;
```

Then prompt Cortex Code:
```
I have CSV files on my stage @PS_DEMO.YOUR_NAME_HOL.HOL_STAGE.
Create tables from each file using COPY INTO with infer_schema.
Use the filename as the table name.
```

### Step 3: Verify your tables loaded

In [ ]:
USE DATABASE PS_DEMO;
USE SCHEMA YOUR_NAME_HOL;
-- You should see ~37 tables after loading
SHOW TABLES IN SCHEMA YOUR_NAME_HOL

-- Quick row count check on key tables
SELECT 'DIM_CAMPAIGN' T, COUNT(*) N FROM DIM_CAMPAIGN
UNION ALL SELECT 'RAW_META_ADS_INSIGHTS', COUNT(*) FROM RAW_META_ADS_INSIGHTS
UNION ALL SELECT 'SOCIAL_REDDIT_POSTS', COUNT(*) FROM SOCIAL_REDDIT_POSTS
UNION ALL SELECT 'FACT_PSPLUS_SUBS', COUNT(*) FROM FACT_PSPLUS_SUBS;

---

## Section 1: Reorganize (30 min)

**Goal**: Turn 11 raw ad-platform tables + messy agency uploads into a harmonized analyst-ready layer.

**Deliverables**: `PAID_MEDIA_HARMONIZED`, `OWNED_CHANNELS_UNIFIED`, `SOCIAL_FEED`

---

### PROMPT FOR CORTEX CODE (copy/paste this)

```
I'm a paid media analyst at PlayStation. I need one harmonized daily table that lets me compare any channel (Meta, TikTok, Google SEM, YouTube Ads, Reddit, Snap, X, DV360, Hulu CTV, Roku CTV, Twitch) on apples-to-apples metrics.

Look at my schema for the raw tables (they all start with RAW_). Each platform uses slightly different column names and has platform-specific fields. Also include the AGENCY_MANUAL_UPLOADS table which has messy data (string spend, comma-separated impressions, mixed-case campaign names).

Build a CREATE TABLE AS statement called PAID_MEDIA_HARMONIZED with these common columns:
- REPORT_DATE, PLATFORM_SOURCE, CAMPAIGN_NAME, SPEND_USD, IMPRESSIONS, CLICKS, VIDEO_VIEWS (null where not applicable)

IMPORTANT: Normalize PLATFORM_SOURCE to UPPER_SNAKE_CASE (e.g. META, TIKTOK, GOOGLE_SEM, GOOGLE_ADS_YT, REDDIT_ADS, SNAP, X_ADS, DV360, HULU_CTV, ROKU_CTV, TWITCH, AGENCY_MANUAL). Also apply UPPER() and TRIM() to the CAMPAIGN_NAME column so joins to DIM_CAMPAIGN work cleanly in later sections.

Flag any data quality issues you find in the agency manual uploads.
```

In [ ]:
USE DATABASE PS_DEMO;
USE SCHEMA YOUR_NAME_HOL;
-- VALIDATION: After Cortex Code builds your table, run this to verify
SELECT PLATFORM_SOURCE, COUNT(*) AS ROW_COUNT, ROUND(SUM(SPEND_USD)/1e6, 2) AS SPEND_MM
FROM PAID_MEDIA_HARMONIZED
GROUP BY 1 ORDER BY SPEND_MM DESC;

### PROMPT FOR CORTEX CODE: Owned channels

```
Now I need a unified owned-channels daily fact table. In #PS_DEMO.YOUR_NAME_HOL there are: FACT_WEB_SESSIONS, FACT_EMAIL_SENDS, FACT_STORE_EVENTS, FACT_APP_EVENTS, and FACT_OWNED_SOCIAL_DAILY.

Build OWNED_CHANNELS_UNIFIED in my schema that gives me a single daily view across all owned touchpoints with: date, channel_type, metric_name, metric_value (tall/long format) so I can pivot or filter by channel without writing a 5-way join every time.
```

In [ ]:
USE DATABASE PS_DEMO;
USE SCHEMA YOUR_NAME_HOL;
-- VALIDATION
SELECT CHANNEL_TYPE, COUNT(*) AS ROW_COUNT
FROM PS_DEMO.YOUR_NAME_HOL.OWNED_CHANNELS_UNIFIED
GROUP BY 1 ORDER BY 1;

### PROMPT FOR CORTEX CODE: Social feed

```
Combine the Reddit posts and YouTube comments into one SOCIAL_FEED table in my schema. Common columns: date, platform (REDDIT or YOUTUBE), author_pseudonym, comment_text, campaign_id, title_id, existing_sentiment_score.
```

### SELF-CRITIQUE PROMPT

```
Review the three tables you just built (PAID_MEDIA_HARMONIZED, OWNED_CHANNELS_UNIFIED, SOCIAL_FEED). As a marketing analyst would:
- Are there grain mismatches that would cause double-counting?
- Any join keys that won't work with the dimension tables?
- Columns a marketing analyst would expect but don't exist?
Tell me what's wrong before I build reporting on top of it.
```

---

## Section 2: Report (30 min)

**Goal**: Build reporting views from your harmonized layer that a CMO can consume weekly.

**Deliverables**: `RPT_PAID_PERFORMANCE`, `RPT_OWNED_FUNNEL`, `RPT_FOLLOWER_VELOCITY`

---

### PROMPT FOR CORTEX CODE: Paid performance

```
I'm building the CMO's weekly paid performance view. Using my PAID_MEDIA_HARMONIZED table and FACT_PAID_CONVERSIONS and DIM_CAMPAIGN:

Build a view called RPT_PAID_PERFORMANCE with:
- Grain: date x campaign x platform_source
- Metrics: spend, impressions, clicks, CTR, CPM, conversions, attributed_revenue, ROAS, CPA
- Add 7-day rolling averages for spend and ROAS
- Add week-over-week percent change for spend and conversions

Make it a VIEW so it always reflects fresh data.
```

In [ ]:
USE DATABASE PS_DEMO;
USE SCHEMA YOUR_NAME_HOL;
-- VALIDATION: Spot-check ROAS by campaign
SELECT CAMPAIGN_NAME, PLATFORM_SOURCE,
  ROUND(SUM(ATTRIBUTED_REVENUE) / NULLIF(SUM(SPEND_USD), 0), 2) AS ROAS
FROM RPT_PAID_PERFORMANCE
GROUP BY 1, 2
ORDER BY ROAS DESC LIMIT 10;

### PROMPT FOR CORTEX CODE: Owned funnel

```
Build RPT_OWNED_FUNNEL as a view that shows the daily session-to-purchase funnel using FACT_WEB_SESSIONS, FACT_STORE_EVENTS, and FACT_EMAIL_SENDS.

I want to see: sessions -> signups -> store_views -> add_to_cart -> purchases with conversion rates between each stage, by date. Also show email open rate and click rate as parallel owned touchpoints.
```

### PROMPT FOR CORTEX CODE: Follower velocity

```
Build RPT_FOLLOWER_VELOCITY as a view comparing daily paid impressions (from PAID_MEDIA_HARMONIZED) vs owned social impressions (from FACT_OWNED_SOCIAL_DAILY) per channel family, alongside follower net adds from FACT_FOLLOWER_GROWTH_DAILY.

I want to spot: does paid spend amplify organic follower growth, or are they independent? Show correlation-ready columns.
```

---

## Section 3: Sense (30 min)

**Goal**: Enrich social data with AI-powered sentiment (by subtopic), theme classification, and consumer intent signals.

**Deliverables**: `SOCIAL_ASPECT_SENTIMENT`, `SOCIAL_THEMES`, `SOCIAL_INTENT_SIGNALS`

---

### Understanding AI_SENTIMENT with subtopics

The key unlock: `AI_SENTIMENT` can analyze sentiment about **specific aspects** within a comment. Instead of just +0.3 (overall positive), you get:
- Price: -0.82
- Performance: +0.65
- Design: +0.72

This tells you *what* people are reacting to, not just *how* they feel overall.

---

### PROMPT FOR CORTEX CODE: Aspect-level sentiment

```
Using my SOCIAL_FEED table, I want to run aspect-level sentiment analysis on the comments.

For each comment, analyze sentiment about these specific aspects:
- price
- performance
- exclusivity
- multiplayer
- story
- store_experience

Use AI_SENTIMENT with the categories parameter. Create a table called SOCIAL_ASPECT_SENTIMENT with columns: comment_id, platform, campaign_id, title_id, aspect, sentiment_score.

Start with a sample of 1000 rows to validate the output shape, then I'll decide if we scale to the full table.
```

In [ ]:
USE DATABASE PS_DEMO;
USE SCHEMA YOUR_NAME_HOL;
-- VALIDATION: Check aspect sentiment distribution
SELECT ASPECT,
  COUNT(*) AS N,
  ROUND(AVG(SENTIMENT_SCORE), 3) AS AVG_SENT,
  ROUND(AVG(CASE WHEN SENTIMENT_SCORE > 0.2 THEN 1 ELSE 0 END)*100, 1) AS PCT_POS,
  ROUND(AVG(CASE WHEN SENTIMENT_SCORE < -0.2 THEN 1 ELSE 0 END)*100, 1) AS PCT_NEG
FROM SOCIAL_ASPECT_SENTIMENT
GROUP BY 1 ORDER BY AVG_SENT;

### PROMPT FOR CORTEX CODE: Theme classification

```
Using my SOCIAL_FEED table, classify each comment into one of these themes using AI_CLASSIFY:
- Price, Exclusivity, Performance, Hype, Bug/Complaint, Multiplayer, Nostalgia, Store UX, Other

Create SOCIAL_THEMES with: comment_id, platform, classified_theme, confidence_score. Run on 1000 rows first.
```

### PROMPT FOR CORTEX CODE: Consumer intent extraction

```
For each comment in my SOCIAL_FEED that mentions a specific game title, use AI_COMPLETE (or TRY_COMPLETE for error handling) with llama3.1-70b to extract a structured purchase-intent signal.

Return a JSON with:
- intent_label: one of [will_buy, considering, wait_for_reviews, will_not_buy, already_own, not_applicable]
- confidence: 0.0 to 1.0
- title_mentioned: the game title referenced

Create SOCIAL_INTENT_SIGNALS with the parsed fields. Use TRY_PARSE_JSON to handle any malformed responses gracefully. Sample 500 rows.
```

In [ ]:
USE DATABASE PS_DEMO;
USE SCHEMA YOUR_NAME_HOL;
-- VALIDATION: Intent distribution
SELECT INTENT_LABEL, COUNT(*) AS N,
  ROUND(AVG(CONFIDENCE), 2) AS AVG_CONF
FROM PS_DEMO.YOUR_NAME_HOL.SOCIAL_INTENT_SIGNALS
WHERE INTENT_LABEL != 'not_applicable'
GROUP BY 1 ORDER BY N DESC;

### INSIGHT PROMPT (after building all three)

```
Now aggregate my SOCIAL_ASPECT_SENTIMENT and SOCIAL_INTENT_SIGNALS by campaign_id. Join back to my RPT_PAID_PERFORMANCE view.

Show me: for each campaign, where paid spend is high but the aspect sentiment on 'price' is negative - are we spending into a headwind? Which campaigns have strong 'will_buy' intent but low spend?
```

---

## Section 4: Build a Marketing Mix Model (20 min)

**Goal**: Build a real statistical MMM that proves which channels drive PS Plus subscriber growth.

**Deliverables**: `MMM_MODEL_INPUT` table, OLS regression results with p-values, Streamlit dashboard with budget optimizer

This is where it all comes together. You'll ask Cortex Code to build a real regression model on the data you created in Sections 1-3.

---

### PROMPT FOR CORTEX CODE: Build the MMM

```
I want to understand what's actually driving PS Plus subscriber growth.

Using my PAID_MEDIA_HARMONIZED table (which has daily spend by channel) and
FACT_PSPLUS_SUBS (daily subscriber adds/cancels by tier)
and DIM_DATE (weekends, holidays, marketing moments)
and my OWNED_CHANNELS_UNIFIED and SOCIAL_ASPECT_SENTIMENT tables:

1. Build a model input table called MMM_MODEL_INPUT at daily grain with:
   - Dependent variable: total daily PS Plus NET subscriber adds (gross_adds - cancels)
   - Independent variables: per-channel spend (one column per platform), web signups,
     owned social engagement, average sentiment score, plus controls (is_weekend, is_holiday, has_marketing_moment)

2. Then write a Python cell (using statsmodels) that:
   - Applies ADSTOCK transformation to spend columns (geometric decay, rate=0.7)
     to capture lagged media effects
   - Runs OLS regression predicting net_adds from adstocked spend + signals + controls
   - Prints the full model summary: R-squared, F-statistic, and each coefficient with p-value
   - Calculates channel contribution (% of explained variance per channel)
   - Identifies which channels are statistically significant at p<0.05

This is a Marketing Mix Model. I want to see real statistical significance,
not just correlations. Show me which channels actually move the needle.
```

In [ ]:
USE DATABASE PS_DEMO;
USE SCHEMA YOUR_NAME_HOL;
-- VALIDATION: Check your model input table has ~181 rows and all columns populated
SELECT COUNT(*) AS ROW_COUNT,
  COUNT_IF(SPEND_META > 0) AS HAS_META,
  COUNT_IF(NET_ADDS > 0) AS HAS_SUBS,
  ROUND(AVG(NET_ADDS)) AS AVG_NET_ADDS
FROM PS_DEMO.YOUR_NAME_HOL.MMM_MODEL_INPUT;

### PROMPT FOR CORTEX CODE: Visualize as a HTML Dashboard

```
Now take the MMM model results and build a HTML Dashboard that visualizes them.
The dashboard should have three tabs:

1. Channel Contribution: horizontal bar chart showing each channel's % contribution
   to subscriber growth. Color positive channels blue and negative channels red.

2. Model Coefficients: table with channel, coefficient, p-value, and a
   significance flag (highlighted green if p<0.05, red if not).
   Include an interpretation section explaining what the coefficients mean.

3. Budget Optimizer: sliders for each channel's 6-month spend allocation.
   As I move the sliders, show predicted daily subscriber lift vs. current baseline,
   total 6-month lift, and budget change. Use the model coefficients to power
   the predictions.

Use the actual coefficients and p-values from the model we just ran.
Make it interactive and visually polished.
```

### What you just accomplished

In 20 minutes you:
- Built a real statistical model (OLS with adstock decay)
- Proved which channels significantly drive subscriber growth (p<0.05)
- Identified channels where spend correlates with LOWER growth (possible saturation)
- Created an interactive budget optimizer backed by the model

This is a **real Marketing Mix Model** — the same methodology agencies charge $200K+ to build over 3-6 months.

---

## Debrief

In ~2 hours you:

1. **Reorganized** 11 raw platform tables into a harmonized analyst layer
2. **Built** reporting views with ROAS, CPM, funnel conversion, follower velocity
3. **Enriched** social data with aspect-level AI sentiment, theme classification, and intent extraction
4. **Built a real Marketing Mix Model** proving which channels drive PS Plus subscriber growth (with statistical significance)
5. **Deployed a Streamlit dashboard** with an interactive budget optimizer

### Key takeaway

The prompts are portable. Swap `PS_DEMO.HOL_DEMOS` for your own database. The pattern works on any marketing dataset in Snowflake.

### What would have taken a team of 3-5 people months:
- Data engineering: harmonize 11 platforms → 30 min
- Analytics engineering: reporting views → 30 min
- Data science: AI enrichment + sentiment → 30 min
- Quantitative modeling: full MMM + interactive dashboard → 20 min

**Total: 2 hours. One person. Natural language.**

### Next steps
- Save your schema — it persists for 30 days
- Try running the same prompts on your real data
- The MMM methodology scales: more channels, longer time windows, different DVs